## Imports og tilkobling

In [1]:
import sys
import os
sys.path.append('../../../src')
import feature_engineering.værdata as vd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "processeddata"

## Finn og velg værstasjon

In [2]:
# Timenes koordinater
timenes_coords = (58.17792, 8.11984)

# Velg stasjon
nearest_station, stations_df = vd.finn_naermeste_stasjon(*timenes_coords, stasjon_index=1)
print("Henter data fra stasjon:", nearest_station)

Henter data fra stasjon: SN39040


## Definer periode

In [3]:
vinter_periods = [
    ("2024-11-03T23:00:00Z", "2024-11-30T23:00:00Z"),
    ("2024-12-01T00:00:00Z", "2024-12-31T23:00:00Z"),
    ("2025-01-01T00:00:00Z", "2025-01-31T23:00:00Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:00:00Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:00:00Z"),
    ("2026-01-01T00:00:00Z", "2026-01-17T22:00:00Z")
]

## Hent time for time værdata

In [4]:
weather_df_hourly = vd.fetch_weather_periods_hourly(nearest_station, vinter_periods)
weather_df_hourly.head()

,timestamp,air_temperature,wind_speed,precipitation_mm
0,2024-11-03 23:00:00,6.1,1.9,0.0
1,2024-11-04 00:00:00,4.9,3.0,0.0
2,2024-11-04 01:00:00,4.8,3.0,0.0
3,2024-11-04 02:00:00,4.8,3.0,0.0
4,2024-11-04 03:00:00,4.8,1.9,0.0


## Last opp til Azure Blob

In [5]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
weather_df_hourly.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
output_blob_name = "TIMENES_weather.csv"
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

TIMENES_weather.csv lastet opp til blob!
